# Mini-Project 1: House Price Regression

## Goal
Predict house prices using regression models. Practice the **full ML workflow**:

**EDA** --> **Feature Engineering** --> **Model Selection** --> **Evaluation**

## Dataset
We use scikit-learn's **California Housing** dataset (~20,000 samples, 8 features).
Each row represents a census block group in California; the target is the **median house value** (in $100,000s).

## What You Will Practice
- Exploratory data analysis and visualization
- Building baseline models and iterating
- Comparing linear vs. tree-based regressors
- Using pipelines for clean, reproducible workflows
- Evaluating with RMSE, MAE, and R-squared

**Estimated time: ~2 hours**

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

from shared.viz_utils import setup_style, save_fig
setup_style()

print("Setup complete!")

## Load the Data

Load the California Housing dataset and convert it to a pandas DataFrame for easier exploration.

In [ ]:
# Load dataset
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target  # target: median house value in $100k

print(f"Dataset shape: {df.shape}")
print(f"\nFeature descriptions:")
print(housing.DESCR[:1000])
print("...")

df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

## Results Tracker

We will train several models and compare them. This helper function logs RMSE, MAE, and R-squared for each experiment so we can compare at the end.

In [ ]:
results = {}

def log_result(name, y_true, y_pred):
    """Log regression metrics for a model experiment."""
    results[name] = {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R²': r2_score(y_true, y_pred)
    }
    print(f"{name}: RMSE={results[name]['RMSE']:.4f}, MAE={results[name]['MAE']:.4f}, R²={results[name]['R²']:.4f}")

---
## Part 1: Quick EDA

Before modeling, understand your data! Complete the TODO cells below:

1. **Target distribution** -- Is it normally distributed? Skewed? Are there outliers?
2. **Correlation heatmap** -- Which features correlate most with the target?
3. **Scatter plots** -- Visualize the relationship between top features and the target.

> **Tip:** Look for features with high absolute correlation to MedHouseVal. Also check for multicollinearity between features.

In [ ]:
# TODO: Plot the distribution of the target variable (MedHouseVal)
# Hint: Use plt.hist() or sns.histplot()
# Question: Do you notice anything at the upper end of the distribution?

fig, ax = plt.subplots(figsize=(8, 5))

# --- YOUR CODE HERE ---


ax.set_xlabel('Median House Value ($100k)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Target Variable')
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Create a correlation heatmap
# Hint: Use df.corr() and sns.heatmap()
# Which features have the strongest correlation with MedHouseVal?

fig, ax = plt.subplots(figsize=(10, 8))

# --- YOUR CODE HERE ---
# corr_matrix = ...
# sns.heatmap(corr_matrix, ...)


ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# TODO: Scatter plots of top 3 correlated features vs MedHouseVal
# Hint: After computing correlations, pick the top 3 features by absolute correlation
# Use plt.subplots(1, 3, ...) for a clean layout

# --- YOUR CODE HERE ---
# top_features = [...]  # list of 3 feature names

# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for i, feat in enumerate(top_features):
#     axes[i].scatter(df[feat], df['MedHouseVal'], alpha=0.1, s=5)
#     axes[i].set_xlabel(feat)
#     axes[i].set_ylabel('MedHouseVal')
#     axes[i].set_title(f'{feat} vs Target')
# plt.tight_layout()
# plt.show()

## Train/Test Split

Split the data 80/20. We use `random_state=42` for reproducibility.

In [ ]:
# Prepare features and target
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples")

---
## Part 2: Baseline -- Linear Regression

**Always start with the simplest model.** A plain linear regression gives us a baseline to beat. If a fancy model cannot beat this, something is wrong.

Steps:
1. Fit `LinearRegression` on the training data
2. Predict on the test data
3. Log results using our `log_result()` helper

In [ ]:
# TODO: Fit LinearRegression on X_train, y_train
# TODO: Predict on X_test
# TODO: Log results with log_result('Linear Regression', y_test, y_pred)

# --- YOUR CODE HERE ---
# model = LinearRegression()
# model.fit(...)
# y_pred = model.predict(...)
# log_result('Linear Regression', y_test, y_pred)

---
## Part 3: Add Scaling

Feature scaling (standardization) ensures all features have mean=0 and std=1. This is **essential** for:
- **Regularized models** (Ridge, Lasso) -- penalties are applied to coefficients, so features on different scales get penalized unfairly
- **Gradient-based optimizers** -- converge faster with scaled features

Plain linear regression is not affected by scaling (same predictions), but we build the habit of using `Pipeline` for clean workflows.

> **Key concept:** `Pipeline` ensures scaling is fit on training data only and applied consistently to test data, preventing data leakage.

In [ ]:
# TODO: Create a Pipeline with StandardScaler + LinearRegression
# TODO: Fit and predict
# TODO: Log results as 'Scaled Linear'

# --- YOUR CODE HERE ---
# pipe = Pipeline([
#     ('scaler', StandardScaler()),
#     ('model', LinearRegression())
# ])
# pipe.fit(...)
# y_pred = pipe.predict(...)
# log_result('Scaled Linear', y_test, y_pred)

---
## Part 4: Regularization (Ridge and Lasso)

Regularization adds a penalty to the loss function to prevent overfitting:

- **Ridge (L2):** Shrinks coefficients toward zero but keeps all features. Good when many features contribute.
- **Lasso (L1):** Can drive coefficients exactly to zero, performing feature selection. Good when you suspect some features are irrelevant.

The `alpha` hyperparameter controls the regularization strength:
- `alpha=0` --> plain linear regression (no penalty)
- Large `alpha` --> heavy penalty, simpler model (risk underfitting)

**Your task:** Try both Ridge and Lasso, then experiment with different alpha values.

In [ ]:
# TODO: Pipeline with StandardScaler + Ridge(alpha=1.0)
# TODO: Log results as 'Ridge'

# --- YOUR CODE HERE ---


# TODO: Pipeline with StandardScaler + Lasso(alpha=0.1)
# TODO: Log results as 'Lasso'

# --- YOUR CODE HERE ---

In [ ]:
# TODO: Try 3 different alpha values for Ridge, pick the best
# Hint: Use a loop over alpha values like [0.01, 0.1, 1.0, 10.0, 100.0]
# Hint: You can use cross_val_score for a more robust comparison

# --- YOUR CODE HERE ---
# alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
# for alpha in alphas:
#     pipe = Pipeline([
#         ('scaler', StandardScaler()),
#         ('model', Ridge(alpha=alpha))
#     ])
#     scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
#     rmse = np.sqrt(-scores.mean())
#     print(f"  Ridge(alpha={alpha:>6}): CV RMSE = {rmse:.4f}")

---
## Part 5: Tree-Based Models

Tree-based models (Random Forest, Gradient Boosting) are powerful alternatives:

- **No scaling needed** -- trees split on thresholds, so feature scale does not matter
- **Handle non-linear relationships** naturally
- **Feature importances** built in
- Often achieve strong performance with minimal tuning

**Your task:** Fit a Random Forest and a Gradient Boosting regressor.

In [ ]:
# TODO: Fit RandomForestRegressor(n_estimators=100, random_state=42)
# TODO: Log results as 'Random Forest'

# --- YOUR CODE HERE ---


# TODO: Fit GradientBoostingRegressor(n_estimators=100, random_state=42)
# TODO: Log results as 'Gradient Boosting'

# --- YOUR CODE HERE ---

---
## Part 6: Feature Engineering

Can we improve performance by creating new features? **Polynomial features** create interaction terms and squared terms from existing features.

- `PolynomialFeatures(degree=2, interaction_only=True)` creates only interaction terms (e.g., feature_A * feature_B)
- With 8 features, this adds many new columns -- be mindful of computation time

> **Warning:** This may be slow! If it takes too long, try `interaction_only=True` or select fewer features first.

In [ ]:
# TODO: Pipeline with StandardScaler + PolynomialFeatures(degree=2, interaction_only=True) + Ridge
# Warning: This may be slow with many features!
# TODO: Log results as 'Poly + Ridge'

# --- YOUR CODE HERE ---
# pipe = Pipeline([
#     ('scaler', StandardScaler()),
#     ('poly', PolynomialFeatures(degree=2, interaction_only=True)),
#     ('model', Ridge(alpha=1.0))
# ])
# pipe.fit(X_train, y_train)
# y_pred = pipe.predict(X_test)
# log_result('Poly + Ridge', y_test, y_pred)

---
## Part 7: Compare All Models

Run this cell after completing all the TODO sections above. It will create a summary table and bar charts comparing every model you trained.

In [ ]:
# This cell is pre-filled -- just run it after completing the TODOs above
results_df = pd.DataFrame(results).T.sort_values('RMSE')
print("=" * 60)
print("MODEL COMPARISON (sorted by RMSE, lower is better)")
print("=" * 60)
print(results_df.to_string())
print()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(['RMSE', 'MAE', 'R²']):
    results_df[metric].plot(kind='barh', ax=axes[i], edgecolor='black')
    axes[i].set_title(metric)
    axes[i].set_xlabel(metric)
plt.suptitle('Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Part 8: Best Model Deep Dive

Take your best-performing model and create three diagnostic plots:

1. **Actual vs. Predicted** scatter plot -- points should cluster along the diagonal
2. **Residuals distribution** -- should be roughly normal, centered at zero
3. **Feature importances** (if using a tree-based model) -- which features matter most?

In [ ]:
# TODO: Take your best model and create diagnostic plots
# Re-fit or reuse the best model from above

# --- YOUR CODE HERE ---
# best_model = ...  # your best model (e.g., GradientBoostingRegressor)
# best_model.fit(X_train, y_train)
# y_pred_best = best_model.predict(X_test)

# fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# # Plot 1: Actual vs Predicted
# axes[0].scatter(y_test, y_pred_best, alpha=0.3, s=10)
# axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
# axes[0].set_xlabel('Actual')
# axes[0].set_ylabel('Predicted')
# axes[0].set_title('Actual vs Predicted')

# # Plot 2: Residuals distribution
# residuals = y_test - y_pred_best
# axes[1].hist(residuals, bins=50, edgecolor='black')
# axes[1].axvline(x=0, color='r', linestyle='--')
# axes[1].set_xlabel('Residual')
# axes[1].set_ylabel('Count')
# axes[1].set_title('Residuals Distribution')

# # Plot 3: Feature importances (for tree-based models)
# if hasattr(best_model, 'feature_importances_'):
#     importances = pd.Series(best_model.feature_importances_, index=X.columns)
#     importances.sort_values().plot(kind='barh', ax=axes[2], edgecolor='black')
#     axes[2].set_title('Feature Importances')
# else:
#     axes[2].text(0.5, 0.5, 'N/A for this model type',
#                  ha='center', va='center', transform=axes[2].transAxes)
#     axes[2].set_title('Feature Importances')

# plt.tight_layout()
# plt.show()

---
## What I Learned

Fill in your takeaways after completing the project:

- **Best model:** _____ with RMSE = _____ and R-squared = _____
- **Scaling impact:** Scaling (did / did not) significantly change linear regression results because _____
- **Regularization:** Ridge/Lasso (did / did not) improve over plain linear regression. The best alpha was _____
- **Linear vs. Tree-based:** Tree-based models performed (better / worse / similar) because _____
- **Feature engineering:** Polynomial features (helped / did not help) because _____
- **Most important features:** The top 3 features were _____, _____, _____
- **Surprise finding:** _____
- **Next steps I would try:** _____